# Week 20 workbook --- GRPO, RLOO, RLVR

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)

import numpy as np
import matplotlib.pyplot as plt
import torch

from project import solutions as S
from project.env import exact_match_reward, extract_final_number, ArithmeticTask
from project.train import grpo_advantages, rloo_advantages, train

## Exercise 1 ($\bigstar$). Extract the final integer from a completion.

Write a function that mirrors the GSM8K answer-extraction convention:
given an arbitrary text completion, return the last integer (positive
or negative) appearing in the text, or `None` if no integer is present.

> **Hint** (try yourself first): Python's `re.findall(r'-?\d+', text)` returns every signed-integer match in order; the last element is what you want.

In [ ]:
def exercise_1(text):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Hidden test --- delegates to project.solutions.
ref = S.exercise_1
for t, expected in [("the answer is 17", 17), ("3 then 4 so 7", 7), ("nope", None), ("-2 + 3 = 1", 1)]:
    assert ref(t) == expected
print('exercise 1 ok')

## Exercise 2 ($\bigstar\bigstar$). RLOO advantage from a vector of group rewards.

Given a length-$G$ vector of rewards, return the length-$G$ vector of
RLOO advantages
$$\hat A_i = r_i - \frac{1}{G-1}\sum_{j \neq i} r_j = \frac{G}{G-1}(r_i - \bar r).$$

> **Hint**: $\sum_{j \neq i} r_j = (\sum_j r_j) - r_i$; the closed-form $G/(G-1)\,(r_i - \bar r)$ is the cleanest.

In [ ]:
def exercise_2(rewards):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_2
a = ref(np.array([1.0, 0.0, 1.0, 0.0]))
assert abs(a.sum()) < 1e-9, "RLOO advantages should sum to zero on symmetric input"
assert np.allclose(ref([1.0, 1.0, 1.0]), [0.0, 0.0, 0.0]), "constant rewards -> zero advantage"
print('exercise 2 ok')

## Exercise 3 ($\bigstar\bigstar$). GRPO advantage --- standard and Dr.GRPO variants.

Given a batch of group rewards $r \in \mathbb{R}^{B \times G}$, return
$(r_i - \bar r) / (\sigma_r + \varepsilon)$ (standard) or just
$(r_i - \bar r)$ (Dr.GRPO debiased).

In [ ]:
def exercise_3(rewards, debias=False, eps=1e-6):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_3
r = np.array([[0.0, 1.0, 0.0, 1.0]])
a = ref(r)
assert abs(a.mean()) < 1e-6
assert abs(a.std() - 1.0) < 1e-2
ad = ref(r, debias=True)
assert abs(ad.mean()) < 1e-12
print('exercise 3 ok')

## Exercise 4 ($\bigstar\bigstar$). Coefficient variance diagnostic.


Sample groups of Bernoulli rewards and compare raw-reward variance with within-group centred variance. This measures coefficients, not full policy-gradient variance. The self-including mean is action dependent; RLOO uses the other samples and multiplies centred coefficients by G/(G-1).
Sample $B$ groups of $G$ Bernoulli$(p)$ rewards and confirm
$\Var[r_i - \bar r] \leq \Var[r_i]$ --- the variance reduction lemma in
the notes.

In [ ]:
def exercise_4(rewards):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_4
rng = np.random.default_rng(0)
r = (rng.uniform(size=(500, 8)) < 0.3).astype(float)
v_r, v_c = ref(r)
assert v_c <= v_r + 1e-9, f"baseline failed: {v_c} > {v_r}"
print(f'exercise 4 ok --- v_reinforce={v_r:.4f}  v_centered={v_c:.4f}')

## Exercise 5 ($\bigstar\bigstar$). KL k3 estimator.

Schulman's low-variance unbiased KL estimator: given paired
log-probabilities $\log p$ and $\log q$ for samples drawn from $p$,
$k_3 = \exp(\log q - \log p) - (\log q - \log p) - 1$. The sample
mean of $k_3$ is unbiased for $\KL(p \,\|\, q)$ and always non-negative.

In [ ]:
def exercise_5(logp, logq):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_5
rng = np.random.default_rng(1)
logp = -rng.uniform(0.1, 1.0, size=200)
assert abs(ref(logp, logp)) < 1e-9
assert ref(logp, logp - 0.5) > 0
print('exercise 5 ok')

## Exercise 6 ($\bigstar\bigstar$). Token-level PPO clipped surrogate.

Return the (negative) clipped surrogate elementwise.

In [ ]:
def exercise_6(ratio, adv, clip_eps=0.2):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_6
loss = ref(np.array([2.0, 0.1, 1.0]), np.array([1.0, 1.0, 1.0]), 0.2)
assert abs(loss[0] - (-1.2)) < 1e-9
print('exercise 6 ok')

## Exercise 7 ($\bigstar\bigstar$). Process-reward aggregation (matching application).

Given a batch of step-reward sequences (think: per-interview-stage
scores in an ATS pipeline), aggregate each row into a single
trajectory score. Implement `min`, `mean`, and `prod` aggregators.
Which is appropriate for the ATS analogue, and why?

In [ ]:
def exercise_7(step_rewards, mode='min'):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_7
s = np.array([[0.9, 0.9, 0.2], [0.8, 0.8, 0.8]])
assert np.allclose(ref(s, 'min'), [0.2, 0.8])
assert np.allclose(ref(s, 'mean'), [0.6666667, 0.8])
assert np.allclose(ref(s, 'prod'), [0.162, 0.512])
print('exercise 7 ok')

## Exercise 8 ($\bigstar\bigstar\bigstar$, open-ended). Reward shaping and format hacking.

Implement a shaped reward that gives partial credit ($+\alpha$) for
emitting *any* parseable integer, plus the binary exact-match. Then
compare the policy's behaviour under the shaped and unshaped reward.
What do you observe? (Hint: count the rate at which the policy emits
*any* number vs the rate at which it emits the *correct* number.)

In [ ]:
def exercise_8(completion, target, fmt_bonus=0.1):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
ref = S.exercise_8
assert ref('3', 3) >= 1.0  # correct gets base + bonus
assert ref('hello', 3) == 0.0  # nothing parseable, no reward
assert 0 < ref('answer 4', 3) < 1.0  # wrong number but parseable
print('exercise 8 ok (open-ended discussion in markdown below)')

## Visualisation --- training curve and variance comparison


The prompts come from the same finite task set used for training; this tests new samples, not generalisation to disjoint prompts.
Run the reference GRPO loop for a short pass and plot fresh-sample accuracy
and the variance-vs-group-size diagnostic. These are the same figures
used in `notes.pdf`.

In [ ]:
result = train(seed=0, num_steps=200, eval_every=50, batch_prompts=4, group_size=8, sft_warmup_steps=400)
h = result['history']
fig, ax = plt.subplots()
ax.plot(h['eval_step'], h['eval_acc'], marker='o', color=RL_PALETTE['blue'])
ax.set_xlabel('step'); ax.set_ylabel('held-out greedy accuracy')
ax.set_title('GRPO accuracy on toy arithmetic')
fig.tight_layout(); plt.show()
print(f"final accuracy: {result['final_acc']:.3f}")

In [ ]:
rng = np.random.default_rng(0)
Gs = [2, 4, 8, 16, 32]
vs = []
for G in Gs:
    r = (rng.uniform(size=(2000, G)) < 0.3).astype(float)
    _, v_c = S.exercise_4(r)
    vs.append(v_c)
fig, ax = plt.subplots()
ax.plot(Gs, vs, marker='o', color=RL_PALETTE['teal'])
ax.set_xscale('log', base=2); ax.set_xlabel('group size G'); ax.set_ylabel('Var[r_i - mean(r)]')
ax.set_title('Centered estimator variance vs G (Bernoulli p=0.3)')
fig.tight_layout(); plt.show()

## Discussion (write your own)

1. How does the GRPO fresh-sample accuracy curve change if you set `sft_warmup_steps=0`? Explain in terms of the cold-start pathology of RLVR.
2. Try `variant='dr_grpo'` in the trainer config. Do you see shorter or longer completions on average? Why might that be?
3. Replace the verifier with `sympy_equiv_reward` from `project/env.py`. For this toy task it changes nothing --- why? In what task would it matter?

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 ($\bigstar$). Extract the final integer from a completion.</summary>

> **Hint** (try yourself first): Python's `re.findall(r'-?\d+', text)` returns every signed-integer match in order; the last element is what you want.

</details>

<details>
<summary>Exercise 2 ($\bigstar\bigstar$). RLOO advantage from a vector of group rewards.</summary>

> **Hint**: $\sum_{j \neq i} r_j = (\sum_j r_j) - r_i$; the closed-form $G/(G-1)\,(r_i - \bar r)$ is the cleanest.

</details>

<details>
<summary>Exercise 3 ($\bigstar\bigstar$). GRPO advantage --- standard and Dr.GRPO variants.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 4 ($\bigstar\bigstar$). Coefficient variance diagnostic.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 5 ($\bigstar\bigstar$). KL k3 estimator.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 6 ($\bigstar\bigstar$). Token-level PPO clipped surrogate.</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 7 ($\bigstar\bigstar$). Process-reward aggregation (matching application).</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 8 ($\bigstar\bigstar\bigstar$, open-ended). Reward shaping and format hacking.</summary>

(Hint: count the rate at which the policy emits
*any* number vs the rate at which it emits the *correct* number.)

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
